# Prerequisite Statistics Knowledge

> **Chapter 1.0 — Foundations**

Before exploring data or building models, you need a firm grasp of descriptive statistics. These are the building blocks that appear in every EDA, every evaluation metric, and every algorithm assumption.

**Concepts covered:**
- Measures of central tendency: Mean, Median, Mode
- Measures of spread: Variance, Standard Deviation, Quartiles, IQR

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# A simple dataset we'll use throughout this notebook
data = [4, 7, 13, 2, 1, 7, 9, 3, 7, 12, 15, 6, 11, 5, 8]
s = pd.Series(data)

print("Dataset :", sorted(data))
print("n       :", len(data))

---

## 1. Measures of Central Tendency

Central tendency tells you **where the center of the data is** — a single representative value for the entire dataset.

---

### 1.1 Mean (Arithmetic Average)

The mean is the sum of all values divided by the count:

$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$$

**Strengths:** Uses all data points, mathematically convenient.  
**Weakness:** Sensitive to outliers — one extreme value pulls it significantly.

> If a dataset has outliers, the mean is a misleading center.

In [ ]:
mean = sum(data) / len(data)
print(f"Sum  : {sum(data)}")
print(f"n    : {len(data)}")
print(f"Mean : {sum(data)}/{len(data)} = {mean:.2f}")
print(f"pandas: {s.mean():.2f}")

### 1.2 Median (Middle Value)

The median is the **middle value** when data is sorted. It splits the dataset into two equal halves.

$$\text{Median} = \begin{cases} x_{\frac{n+1}{2}} & \text{if } n \text{ is odd} \\[6pt] \dfrac{x_{\frac{n}{2}} + x_{\frac{n}{2}+1}}{2} & \text{if } n \text{ is even} \end{cases}$$

**Strengths:** Not affected by outliers — a resistant statistic.  
**Weakness:** Ignores the actual magnitudes of non-central values.

> Prefer the median over the mean when the distribution is skewed (e.g., income, house prices).

In [ ]:
sorted_data = sorted(data)
n = len(sorted_data)

if n % 2 == 1:
    median = sorted_data[n // 2]
    print(f"n is odd → middle index = {n // 2} → median = {median}")
else:
    median = (sorted_data[n // 2 - 1] + sorted_data[n // 2]) / 2
    print(f"n is even → average of indices {n//2-1} & {n//2} → median = {median}")

print(f"Sorted : {sorted_data}")
print(f"pandas : {s.median()}")

### 1.3 Mode (Most Frequent Value)

The mode is the value that appears **most often** in the dataset.

- A dataset can have **no mode** (all values unique), **one mode** (unimodal), or **multiple modes** (bimodal / multimodal).
- The only measure of central tendency applicable to **categorical data**.

$$\text{Mode} = \underset{x}{\arg\max} \ f(x)$$

where $f(x)$ is the frequency of value $x$.

In [ ]:
freq = Counter(data)
max_count = max(freq.values())
mode = [k for k, v in freq.items() if v == max_count]

print(f"Frequencies : {dict(sorted(freq.items()))}")
print(f"Max count   : {max_count}")
print(f"Mode        : {mode}")
print(f"pandas      : {list(s.mode())}")

---

## 2. Measures of Spread (Dispersion)

Spread tells you **how much the data varies** around the center. Two datasets can have the same mean but very different distributions.

---

### 2.1 Variance

Variance is the **average squared deviation** from the mean.

**Population variance** (when you have all data):

$$\sigma^2 = \frac{1}{n} \sum_{i=1}^{n}(x_i - \bar{x})^2$$

**Sample variance** (when your data is a sample — use $n-1$ to correct for bias):

$$s^2 = \frac{1}{n-1} \sum_{i=1}^{n}(x_i - \bar{x})^2$$

The $n-1$ denominator is called **Bessel's correction** — it makes the sample variance an unbiased estimator of the population variance.

> Variance is in **squared units**, which makes it hard to interpret directly. That's why we use standard deviation.

In [ ]:
mean = s.mean()
squared_devs = [(x - mean)**2 for x in data]

pop_variance = sum(squared_devs) / len(data)          # σ²
sample_variance = sum(squared_devs) / (len(data) - 1) # s²

print(f"Mean                  : {mean:.2f}")
print(f"Squared deviations    : {[round(d, 2) for d in squared_devs]}")
print(f"Population variance σ²: {pop_variance:.2f}")
print(f"Sample variance s²    : {sample_variance:.2f}")
print(f"pandas (ddof=1)       : {s.var():.2f}")

### 2.2 Standard Deviation

Standard deviation is simply the **square root of variance**, bringing the unit back to the original scale.

$$\sigma = \sqrt{\sigma^2} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

$$s = \sqrt{s^2} = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

**Interpretation:** ~68% of data falls within $\bar{x} \pm 1\sigma$, ~95% within $\bar{x} \pm 2\sigma$ (for normal distributions — the **empirical rule**).

**Weakness:** Like the mean, SD is pulled by outliers.

In [ ]:
import math

pop_std = math.sqrt(pop_variance)
sample_std = math.sqrt(sample_variance)

print(f"Population std σ : {pop_std:.2f}")
print(f"Sample std s     : {sample_std:.2f}")
print(f"pandas (ddof=1)  : {s.std():.2f}")
print()
print(f"Empirical rule (sample std):")
print(f"  68% range : [{mean - sample_std:.2f}, {mean + sample_std:.2f}]")
print(f"  95% range : [{mean - 2*sample_std:.2f}, {mean + 2*sample_std:.2f}]")

### 2.3 Quartiles

Quartiles divide a **sorted** dataset into four equal parts.

| Quartile | Symbol | Percentile | Meaning |
|---|---|---|---|
| First quartile | $Q_1$ | 25th | 25% of data is below this |
| Second quartile | $Q_2$ | 50th | The median |
| Third quartile | $Q_3$ | 75th | 75% of data is below this |

$$Q_1 = \text{median of the lower half} \qquad Q_3 = \text{median of the upper half}$$

The **five-number summary** — $(\min,\ Q_1,\ Q_2,\ Q_3,\ \max)$ — gives a compact picture of a distribution and is what a box plot displays.

In [ ]:
Q1 = s.quantile(0.25)
Q2 = s.quantile(0.50)
Q3 = s.quantile(0.75)

print(f"Sorted data : {sorted(data)}")
print()
print(f"Min : {s.min()}")
print(f"Q1  : {Q1}")
print(f"Q2  : {Q2}  ← median")
print(f"Q3  : {Q3}")
print(f"Max : {s.max()}")
print()
print("Five-number summary:")
print(s.describe()[["min", "25%", "50%", "75%", "max"]])

### 2.4 Interquartile Range (IQR)

The IQR is the distance between $Q_1$ and $Q_3$ — the spread of the **middle 50%** of the data.

$$\boxed{IQR = Q_3 - Q_1}$$

Because it ignores the bottom 25% and top 25%, it is not influenced by extreme values — making it a **robust measure of spread**.

**Tukey fences** (outlier detection):

$$\text{Lower fence} = Q_1 - 1.5 \times IQR \qquad \text{Upper fence} = Q_3 + 1.5 \times IQR$$

| Measure | Sensitive to outliers? | Unit |
|---|---|---|
| Variance / SD | Yes | Squared / original |
| IQR | No | Original |

> Use **SD** when the distribution is approximately normal and has no extreme values.  
> Use **IQR** when the distribution is skewed or outliers are present.

In [ ]:
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print(f"Q1           : {Q1}")
print(f"Q3           : {Q3}")
print(f"IQR          : {IQR}")
print(f"Lower fence  : {lower_fence:.2f}")
print(f"Upper fence  : {upper_fence:.2f}")

---

## 3. Summary and Comparison

In [ ]:
summary = {
    "Mean"              : round(s.mean(), 2),
    "Median"            : s.median(),
    "Mode"              : list(s.mode()),
    "Variance (sample)" : round(s.var(), 2),
    "Std Dev (sample)"  : round(s.std(), 2),
    "Q1"                : Q1,
    "Q2 (Median)"       : Q2,
    "Q3"                : Q3,
    "IQR"               : IQR,
}

for k, v in summary.items():
    print(f"  {k:<22}: {v}")

# Visual overview
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(data, bins=8, edgecolor="white", color="steelblue")
axes[0].axvline(s.mean(),   color="red",    linestyle="--", label=f"Mean ({s.mean():.1f})")
axes[0].axvline(s.median(), color="orange", linestyle="--", label=f"Median ({s.median():.1f})")
axes[0].axvline(float(s.mode()[0]), color="green", linestyle="--", label=f"Mode ({int(s.mode()[0])})")
axes[0].set_title("Central Tendency")
axes[0].legend()

axes[1].boxplot(data, vert=True, patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.5))
axes[1].set_title(f"Spread  |  IQR = {IQR}  |  SD = {s.std():.2f}")
axes[1].set_xticks([])

plt.tight_layout()
plt.show()